# Part B — CLIP fine-tuning · Shreyas K M

Run this notebook on **Kaggle (GPU accelerator, Internet on)** or
**Google Colab (GPU runtime)**. It detects the platform automatically.
On Kaggle, import this `.ipynb`, select an available GPU such as T4 ×2, enable
Internet, then use **Save Version → Save & Run All** for the final recorded run.
The code uses one GPU (`cuda:0`) even when Kaggle assigns two; no setting changes
are needed. Packages, public CLIP weights and datasets require Internet access.
The team's shared location is the GitHub repository. All training code is embedded
below; cloning the repository is unnecessary. Parts A/C/D can import the same
`src/common.py` after the corrected version is shared.

**QuickGELU correction:** this version explicitly matches the OpenAI weights'
activation function in both encoders. If an earlier run printed a QuickGELU
mismatch warning, preserve its files as preliminary artifacts and restart the
baseline and all four experiments with this notebook. Do not load those older
checkpoints into the corrected architecture. The default output folder below is
new so earlier artifacts are retained. Parts A/C/D must use this corrected loader.

**Scope:** CLIP ViT-B/32 (`open_clip`, `pretrained='openai'`), four base training runs:
Pets; Pets→Flowers; Pets→EuroSAT; Pets→EuroSAT with text frozen. The three branches
each start from `theta_pets.pt`. DTD is evaluation only. The 56-pair grid is Phase II.

**Fixed protocol:** AdamW, lr `1e-5`, batch 64 (final batch may be smaller), 1 epoch,
at most 2000 train/test images, seed 0. Flowers has only **1020 training images**;
use all of them without mixing validation or repeating images. Weight decay is 0.1
from the shared config. No scheduler or added classifier. Use the checkpoint's
deterministic CLIP resize/crop preprocessing for both training and testing.

The objective is cross-entropy between each image and **all class text prototypes**.
For every class, normalize its ten prompt embeddings, average, and normalize again.
Both towers receive gradients, except in the text-frozen control. `logit_scale`
retains its loaded value throughout (it is never set to a new constant).

**Split details:** Pets `train` maps to torchvision `trainval`; test is `test`.
EuroSAT has no official split: generate one seed-0 permutation, take its first
2000 indices for test and next 2000 for train. This preserves the team's existing
EuroSAT evaluation subset and prevents leakage. Flowers/DTD use official splits;
DTD uses partition 1. Each official split is independently shuffled with
`torch.Generator().manual_seed(0)` before taking at most 2000 indices.
The complete selection and class order are saved to `protocol.json`.

The 2000-image cap limits processing, **not download size**: torchvision downloads
the full dataset archives. Initial installs/downloads are outside training time.


In [ ]:
%pip -q install 'open_clip_torch==3.3.0' 'timm==1.0.22' pyyaml pandas scipy tqdm


In [ ]:
from pathlib import Path
import os
import sys

IS_KAGGLE = Path('/kaggle/working').is_dir()
IS_COLAB = not IS_KAGGLE and Path('/content').is_dir()
if IS_KAGGLE:
    PLATFORM = 'Kaggle'
    WORK_DIR = Path('/kaggle/working/encoder-forgetting-signal-partb')
    DATA_DIR = Path('/tmp/encoder-forgetting-data')
    EXPORT_DIR = Path('/kaggle/working')
elif IS_COLAB:
    PLATFORM = 'Colab'
    WORK_DIR = Path('/content/encoder-forgetting-signal-partb')
    DATA_DIR = Path('/content/encoder-forgetting-data')
    EXPORT_DIR = WORK_DIR
else:
    PLATFORM = 'Local Jupyter'
    WORK_DIR = Path.cwd() / '.notebook_partb'
    DATA_DIR = WORK_DIR / 'data'
    EXPORT_DIR = WORK_DIR

WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
for directory in ('src', 'configs'):
    Path(directory).mkdir(exist_ok=True)
Path('src/__init__.py').touch()

# Kaggle needs no Google Drive integration. Save a completed notebook version
# with outputs and download its archive before discarding the session.
# On Colab only, optionally enable Drive for persistence across runtime resets.
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    if not IS_COLAB:
        raise ValueError('Google Drive mounting is supported here only on Colab.')
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = Path('/content/drive/MyDrive/PW26_VK_06/checkpoints_quickgelu')
else:
    OUTPUT_DIR = WORK_DIR / 'artifacts' / 'checkpoints_quickgelu'
print('Platform:', PLATFORM)
print('Dataset cache (not exported on Kaggle):', DATA_DIR)
print('Checkpoints and measured results:', OUTPUT_DIR)


## Shared implementation

These cells materialize the source under `WORK_DIR`. Kaggle checkpoints and
exports stay under `/kaggle/working`, while dataset downloads go to temporary
storage outside the exported output tree. Colab uses its local `/content` disk.
Checkpoints, metrics, and the protocol use `OUTPUT_DIR`.
Expand these cells to inspect the exact training and split implementation.


In [ ]:
%%writefile configs/config.yaml
model:
  name: "ViT-B-32"
  pretrained: "openai"
  logit_scale: 100
  freeze_logit_scale: true
data:
  root: "data/"
  datasets: [OxfordIIITPet, Flowers102, EuroSAT, DTD]
  max_images: 2000
  seed: 0
prompts:
  - "a photo of a {}."
  - "a picture of a {}."
  - "an image of a {}."
  - "a close-up photo of a {}."
  - "a good photo of a {}."
  - "a cropped photo of a {}."
  - "a bright photo of a {}."
  - "a photo of the {}."
  - "a photo of my {}."
  - "a photo of one {}."
finetune:
  optimizer: AdamW
  lr: 1.0e-5
  weight_decay: 0.1
  batch_size: 64
  epochs: 1
runs: [pets, pets_to_flowers, pets_to_eurosat, pets_to_eurosat_textfrozen]
confidence:
  ece_bins: 15
  confident_wrong_threshold: 0.9


In [ ]:
%%writefile src/common.py
"""Shared utilities for datasets, CLIP loading, and feature extraction.

Used across Part A (Signal), Part B (Finetune), Part C (Swap), and Part D (Confidence).
Imports of torch, torchvision, and open_clip are deferred inside functions so that
lightweight maths and tests can run in environments without GPU or torch installed.
"""

from __future__ import annotations

import gc
import os
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import yaml

# Oxford label order; see torchvision/datasets/flowers102.py. Kept locally for
# torchvision releases which do not expose Flowers102.classes.
FLOWERS102_CLASSES: List[str] = [
    "pink primrose", "hard-leaved pocket orchid", "canterbury bells", "sweet pea",
    "english marigold", "tiger lily", "moon orchid", "bird of paradise", "monkshood",
    "globe thistle", "snapdragon", "colt's foot", "king protea", "spear thistle",
    "yellow iris", "globe-flower", "purple coneflower", "peruvian lily", "balloon flower",
    "giant white arum lily", "fire lily", "pincushion flower", "fritillary", "red ginger",
    "grape hyacinth", "corn poppy", "prince of wales feathers", "stemless gentian",
    "artichoke", "sweet william", "carnation", "garden phlox", "love in the mist",
    "mexican aster", "alpine sea holly", "ruby-lipped cattleya", "cape flower",
    "great masterwort", "siam tulip", "lenten rose", "barbeton daisy", "daffodil",
    "sword lily", "poinsettia", "bolero deep blue", "wallflower", "marigold",
    "buttercup", "oxeye daisy", "common dandelion", "petunia", "wild pansy",
    "primula", "sunflower", "pelargonium", "bishop of llandaff", "gaura",
    "geranium", "orange dahlia", "pink-yellow dahlia", "cautleya spicata",
    "japanese anemone", "black-eyed susan", "silverbush", "californian poppy",
    "osteospermum", "spring crocus", "bearded iris", "windflower", "tree poppy",
    "gazania", "azalea", "water lily", "rose", "thorn apple",
    "morning glory", "passion flower", "lotus", "toad lily", "anthurium",
    "frangipani", "clematis", "hibiscus", "columbine", "desert-rose",
    "tree mallow", "magnolia", "cyclamen", "watercress", "canna lily",
    "hippeastrum", "bee balm", "ball moss", "foxglove", "bougainvillea",
    "camellia", "mallow", "mexican petunia", "bromelia", "blanket flower",
    "trumpet creeper", "blackberry lily"
]

# Standard class names for EuroSAT (satellite land-use classes)
EUROSAT_CLASSES: List[str] = [
    "annual crop", "forest", "herbaceous vegetation", "highway", "industrial",
    "pasture", "permanent crop", "residential", "river", "sea lake"
]


def load_config(path: str = "configs/config.yaml") -> Dict[str, Any]:
    """Read project configuration from YAML."""
    if not os.path.exists(path):
        # Fallback to look one directory up if called from scripts or notebooks
        alt_path = os.path.join(os.path.dirname(__file__), "..", path)
        if os.path.exists(alt_path):
            path = alt_path
    with open(path, "r", encoding="utf-8") as handle:
        return yaml.safe_load(handle)


def load_clip(
    ckpt: Optional[str] = None,
    device: str = "cpu",
    cfg: Optional[Dict[str, Any]] = None,
) -> Tuple[Any, Any, Any]:
    """Load CLIP ViT-B/32 in eval mode.

    Returns:
        (model, preprocess_transform, tokenizer)
    """
    import open_clip
    import torch

    cfg = cfg if cfg is not None else load_config()
    model_name = cfg["model"]["name"]
    pretrained = cfg["model"]["pretrained"]

    model, _, preprocess = open_clip.create_model_and_transforms(
        model_name, pretrained=pretrained,
        # OpenCLIP 3.3 does not automatically apply this pretrained-tag setting.
        # OpenAI weights require QuickGELU in BOTH towers, including when loading
        # our own state dict (activation functions are absent from state dicts).
        force_quick_gelu=pretrained == "openai",
    )

    if ckpt is not None:
        state = torch.load(ckpt, map_location="cpu", weights_only=True)
        if isinstance(state, dict):
            for k in ("state_dict", "model"):
                if k in state and isinstance(state[k], dict):
                    state = state[k]
                    break
        clean_state = {k.removeprefix("module."): v for k, v in state.items()}
        model.load_state_dict(clean_state, strict=True)

    model = model.to(device)
    model.eval()
    tokenizer = open_clip.get_tokenizer(model_name)
    return model, preprocess, tokenizer


def free_model(model: Any) -> None:
    """Move model off GPU and release memory."""
    import torch

    model.to("cpu")
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def get_dataset_loader(
    dataset_name: str,
    preprocess: Any,
    cfg: Dict[str, Any],
    split: str = "test",
) -> Tuple[Any, List[str], List[str]]:
    """Build a deterministic DataLoader for any of the 4 benchmark datasets.

    Supports: 'OxfordIIITPet', 'Flowers102', 'EuroSAT', 'DTD'.
    Applies the team standard seed (0) and at most 2000 images per split.
    Pets train maps to trainval. EuroSAT test is the first capped seed-0
    permutation block; train is the next block, with no overlap.

    Returns:
        (DataLoader, list_of_image_ids, list_of_classnames)
    """
    import torch
    from torch.utils.data import DataLoader, Subset
    import torchvision.datasets as tv_datasets

    data_cfg = cfg["data"]
    root = data_cfg["root"]
    max_images = data_cfg.get("max_images", 2000)
    seed = data_cfg.get("seed", 0)

    ds_name_lower = dataset_name.lower().replace("-", "").replace("_", "")

    if split not in {"train", "test"}:
        raise ValueError("Use split='train' or split='test'. Pets train maps to trainval.")
    if not 0 < max_images <= 2000:
        raise ValueError("max_images must be between 1 and 2000.")

    if "pet" in ds_name_lower:
        ds = tv_datasets.OxfordIIITPet(
            root=root, split="trainval" if split == "train" else "test",
            download=True, transform=preprocess
        )
        classnames = [name.replace("_", " ") for name in ds.classes]

    elif "flower" in ds_name_lower:
        ds = tv_datasets.Flowers102(
            root=root, split=split, download=True, transform=preprocess
        )
        classnames = FLOWERS102_CLASSES

    elif "eurosat" in ds_name_lower:
        ds = tv_datasets.EuroSAT(
            root=root, download=True, transform=preprocess
        )
        classnames = EUROSAT_CLASSES

    elif "dtd" in ds_name_lower:
        ds = tv_datasets.DTD(
            root=root, split=split, download=True, transform=preprocess
        )
        classnames = [name.replace("_", " ") for name in ds.classes]

    else:
        raise ValueError(f"Unknown dataset name: {dataset_name}. Expected Pets, Flowers102, EuroSAT, or DTD.")

    # Apply deterministic shuffling & limit to max_images
    generator = torch.Generator().manual_seed(seed)
    total_len = len(ds)
    num_samples = min(total_len, max_images)
    permutation = torch.randperm(total_len, generator=generator)
    # EuroSAT has no official split. Preserve the shared test subset and select
    # training images from the remainder of the SAME seed-0 permutation.
    start = num_samples if "eurosat" in ds_name_lower and split == "train" else 0
    permuted_idx = permutation[start:start + num_samples].tolist()
    if not permuted_idx:
        raise ValueError("No disjoint training images remain after the EuroSAT test holdout.")

    subset = Subset(ds, permuted_idx)
    loader = DataLoader(
        subset, batch_size=cfg.get("finetune", {}).get("batch_size", 64),
        shuffle=False, num_workers=data_cfg.get("num_workers", 2),
        pin_memory=torch.cuda.is_available(),
    )

    prefix = dataset_name if split == "test" else f"{dataset_name}_train"
    image_ids = [f"{prefix}_{idx}" for idx in permuted_idx]
    return loader, image_ids, classnames


def extract_image_features(
    model: Any, loader: Any, device: str
) -> Tuple[np.ndarray, np.ndarray]:
    """Encode all images in loader and return float32 L2-normalized features and labels."""
    import torch

    features: List[np.ndarray] = []
    labels: List[np.ndarray] = []

    with torch.no_grad():
        for batch in loader:
            images = batch[0].to(device)
            targets = batch[1]

            encoded = model.encode_image(images).float()
            encoded = encoded / encoded.norm(dim=-1, keepdim=True)

            features.append(encoded.cpu().numpy().astype(np.float32))
            if isinstance(targets, torch.Tensor):
                labels.append(targets.cpu().numpy())
            else:
                labels.append(np.asarray(targets))

    return np.concatenate(features, axis=0), np.concatenate(labels, axis=0)


def extract_text_features(
    model: Any,
    tokenizer: Any,
    classnames: Sequence[str],
    templates: Sequence[str],
    device: str,
) -> np.ndarray:
    """Build L2-normalized prompt-ensembled text feature vectors for each class.

    For each class, encodes all 10 prompt templates, averages the normalized
    embeddings, and re-normalizes the result.

    Returns:
        [C, d] float32 array.
    """
    import torch

    class_features: List[np.ndarray] = []

    with torch.no_grad():
        for classname in classnames:
            prompts = [template.format(classname) for template in templates]
            tokens = tokenizer(prompts).to(device)
            encoded = model.encode_text(tokens).float()
            encoded = encoded / encoded.norm(dim=-1, keepdim=True)
            mean_feature = encoded.mean(dim=0)
            mean_feature = mean_feature / mean_feature.norm()
            class_features.append(mean_feature.cpu().numpy().astype(np.float32))

    return np.stack(class_features, axis=0)


In [ ]:
%%writefile src/finetune.py
"""Part B: four small sequential CLIP fine-tuning runs, with measured results.

All ten class prompts participate in cross-entropy. Chunked text activation
checkpointing preserves gradients while fitting the Flowers prompts on Colab.
"""
from __future__ import annotations

import copy
import csv
import gc
import hashlib
import importlib.metadata
import json
from pathlib import Path
import random
import time
from typing import Any

from src.common import get_dataset_loader, load_clip

DATASETS = ("OxfordIIITPet", "Flowers102", "EuroSAT", "DTD")
RUNS = (
    ("pets", "OxfordIIITPet", "theta_pets.pt", False),
    ("pets_to_flowers", "Flowers102", "theta_pets_flowers.pt", False),
    ("pets_to_eurosat", "EuroSAT", "theta_pets_eurosat.pt", False),
    ("pets_to_eurosat_textfrozen", "EuroSAT", "theta_pets_eurosat_textfrozen.pt", True),
)


def seed_everything(seed: int) -> None:
    import numpy as np
    import torch

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def configure_trainable(model: Any, text_frozen: bool) -> None:
    """For native CLIP, visual.* is the entire image tower."""
    for name, param in model.named_parameters():
        param.requires_grad_(name.startswith("visual.") or not text_frozen)
    model.logit_scale.requires_grad_(False)
    if getattr(model, "logit_bias", None) is not None:
        model.logit_bias.requires_grad_(False)


def text_fingerprint(model: Any) -> str:
    """Hash text parameters AND buffers, including projection and embeddings."""
    digest = hashlib.sha256()
    for name, value in sorted(model.state_dict().items()):
        if not name.startswith("visual.") and name not in {"logit_scale", "logit_bias"}:
            digest.update(name.encode())
            digest.update(value.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


def prompt_features(model: Any, tokens: Any, n_classes: int, n_templates: int,
                    chunk_size: int = 32, checkpoint_text: bool = False) -> Any:
    """Normalize prompts, average per class, then normalize each class mean.

    Non-reentrant checkpointing keeps parameter gradients with integer inputs.
    Nothing is detached during training.
    """
    import torch
    import torch.nn.functional as F
    from torch.utils.checkpoint import checkpoint

    encoded = []
    for chunk in tokens.split(chunk_size):
        if checkpoint_text and torch.is_grad_enabled():
            part = checkpoint(model.encode_text, chunk, use_reentrant=False)
        else:
            part = model.encode_text(chunk)
        encoded.append(F.normalize(part.float(), dim=-1))
    prompts = torch.cat(encoded).reshape(n_classes, n_templates, -1)
    return F.normalize(prompts.mean(dim=1), dim=-1)


def tokenize_classes(tokenizer: Any, classes: list[str], templates: list[str], device: str) -> Any:
    return tokenizer([t.format(c) for c in classes for t in templates]).to(device)


def _sync(device: str) -> None:
    import torch
    if torch.device(device).type == "cuda":
        torch.cuda.synchronize(device)


def train_model(model: Any, loader: Any, tokenizer: Any, classes: list[str],
                templates: list[str], cfg: dict, device: str,
                text_frozen: bool = False) -> dict:
    import torch
    import torch.nn.functional as F

    settings = cfg["finetune"]
    seed_everything(cfg["data"]["seed"])
    configure_trainable(model, text_frozen)
    model.train()
    if text_frozen:
        model.eval()
        model.visual.train()
    if hasattr(model.visual, "set_grad_checkpointing"):
        model.visual.set_grad_checkpointing(True)
    scale_before = model.logit_scale.detach().clone()
    text_before = text_fingerprint(model) if text_frozen else None
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=float(settings["lr"]), weight_decay=float(settings.get("weight_decay", 0.1)),
    )
    cuda = torch.device(device).type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=cuda)
    tokens = tokenize_classes(tokenizer, classes, templates, device)
    if cuda:
        torch.cuda.reset_peak_memory_stats(device)
    _sync(device)
    started = time.perf_counter()
    cached_text = None
    if text_frozen:
        with torch.no_grad(), torch.autocast(device_type="cuda", enabled=cuda):
            cached_text = prompt_features(model, tokens, len(classes), len(templates))
    loss_sum, correct, count, skipped = 0.0, 0, 0, 0
    for epoch in range(int(settings["epochs"])):
        for step, (images, labels) in enumerate(loader, 1):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", enabled=cuda):
                image_vectors = F.normalize(model.encode_image(images).float(), dim=-1)
                text_vectors = cached_text if text_frozen else prompt_features(
                    model, tokens, len(classes), len(templates), checkpoint_text=True,
                )
            logits = model.logit_scale.exp().float() * image_vectors @ text_vectors.T
            loss = F.cross_entropy(logits, labels)
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite loss; no checkpoint was saved.")
            scaler.scale(loss).backward()
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            skipped += int(scaler.get_scale() < old_scale)
            batch = labels.numel()
            loss_sum += loss.detach().item() * batch
            correct += (logits.argmax(dim=1) == labels).sum().item()
            count += batch
            if step == 1 or step % 10 == 0 or step == len(loader):
                print(f"  epoch {epoch + 1}, batch {step}/{len(loader)}, loss={loss.item():.4f}", flush=True)
    _sync(device)
    seconds = time.perf_counter() - started
    if not torch.equal(scale_before, model.logit_scale.detach()):
        raise AssertionError("logit_scale changed.")
    frozen_verified = text_before == text_fingerprint(model) if text_frozen else None
    if text_frozen and not frozen_verified:
        raise AssertionError("The text-frozen encoder changed.")
    model.eval()
    if hasattr(model.visual, "set_grad_checkpointing"):
        model.visual.set_grad_checkpointing(False)
    return {
        "train_seconds": seconds, "train_minutes": seconds / 60,
        "train_loss": loss_sum / count, "train_online_accuracy_pct": 100 * correct / count,
        "train_images": len(loader.dataset), "optimizer_steps_skipped": skipped,
        "peak_gpu_memory_gb": torch.cuda.max_memory_allocated(device) / 1e9 if cuda else None,
        "text_frozen_verified": frozen_verified, "logit_scale_frozen_verified": True,
        "logit_scale_exp": model.logit_scale.exp().item(),
    }


def evaluate_model(model: Any, loaders: dict, tokenizer: Any, templates: list[str], device: str) -> dict:
    import torch
    import torch.nn.functional as F

    model.eval()
    results = {}
    # Float32 evaluation matches the shared feature-extraction functions.
    with torch.no_grad():
        for name, (loader, _ids, classes) in loaders.items():
            tokens = tokenize_classes(tokenizer, classes, templates, device)
            text = prompt_features(model, tokens, len(classes), len(templates))
            correct, count = 0, 0
            for images, labels in loader:
                images, labels = images.to(device), labels.to(device)
                vectors = F.normalize(model.encode_image(images).float(), dim=-1)
                correct += ((vectors @ text.T).argmax(dim=1) == labels).sum().item()
                count += labels.numel()
            results[f"{name}_accuracy_pct"] = 100 * correct / count
            results[f"{name}_test_images"] = count
            print(f"  {name}: {100 * correct / count:.2f}% ({count} test images)", flush=True)
    return results


def _json_write(path: Path, payload: Any) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, allow_nan=False) + "\n")
    temporary.replace(path)


def _file_hash(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _environment(device: str) -> dict:
    import platform
    import torch

    versions = {}
    for name in ("torch", "torchvision", "open_clip_torch", "numpy"):
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = "not installed"
    return {"python": platform.python_version(), "packages": versions,
            "device": device, "gpu": torch.cuda.get_device_name(device) if device.startswith("cuda") else None}


def run_experiments(cfg: dict, output_dir: str, device: str = "cuda", *,
                    fallback_lr: float | None = None) -> list[dict]:
    """Run/reuse four base runs; optionally add a matched EuroSAT sensitivity pair.

    Reuse requires identical configuration, source code, parent checkpoint hash,
    and checkpoint checksum. Partial/unrecognised artifacts are not overwritten.
    """
    import torch

    if cfg["model"]["name"] != "ViT-B-32" or cfg["model"]["pretrained"] != "openai":
        raise ValueError("This Phase-1 workflow supports ViT-B-32 / openai only.")
    if cfg["data"]["seed"] != 0:
        raise ValueError("The shared protocol requires seed=0.")
    if cfg["finetune"]["optimizer"] != "AdamW":
        raise ValueError("The shared protocol requires AdamW.")
    if float(cfg["finetune"]["lr"]) != 1e-5 or int(cfg["finetune"]["epochs"]) != 1 or cfg["finetune"]["batch_size"] != 64:
        raise ValueError("Base runs require lr=1e-5, epochs=1, batch_size=64.")
    if fallback_lr is not None and fallback_lr != 3e-5:
        raise ValueError("The optional sensitivity learning rate is 3e-5.")
    if device.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError("Select a GPU runtime in Colab before running training.")
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    environment = _environment(device)
    seed_everything(0)
    initial, preprocess, tokenizer = load_clip(device=device, cfg=cfg)
    tests = {name: get_dataset_loader(name, preprocess, cfg, "test") for name in DATASETS}
    trains = {name: get_dataset_loader(name, preprocess, cfg, "train") for name in DATASETS[:3]}
    manifests = {"test": {name: {"ids": value[1], "classes": value[2]} for name, value in tests.items()},
                 "train": {name: {"ids": value[1], "classes": value[2]} for name, value in trains.items()},
                 "eurosat_split": "seed-0 permutation: first max_images test; next max_images train",
                 "pets_train_split": "trainval", "dtd_partition": 1}
    code_hash = hashlib.sha256()
    for filename in ("common.py", "finetune.py"):
        code_hash.update(Path(__file__).with_name(filename).read_bytes())
    protocol = {"cfg": cfg, "manifests": manifests, "code_sha256": code_hash.hexdigest(),
                "packages": environment["packages"]}
    protocol_file = output / "protocol.json"
    if protocol_file.exists() and json.loads(protocol_file.read_text()) != protocol:
        raise ValueError("Output folder contains a different protocol. Choose a new output folder.")
    _json_write(protocol_file, protocol)
    _json_write(output / "environment.json", environment)
    baseline_path = output / "baseline.json"
    if baseline_path.exists():
        baseline = json.loads(baseline_path.read_text())
    else:
        print("Pretrained CLIP baseline", flush=True)
        started = time.perf_counter()
        baseline = {"run": "pretrained", **evaluate_model(initial, tests, tokenizer, cfg["prompts"], device),
                    "eval_seconds": time.perf_counter() - started}
        _json_write(baseline_path, baseline)
    initial.cpu()
    del initial
    gc.collect()
    if device.startswith("cuda"):
        torch.cuda.empty_cache()
    rows = [baseline]

    def run_one(spec: tuple, run_cfg: dict, directory: Path) -> dict:
        name, dataset, filename, frozen = spec
        directory.mkdir(parents=True, exist_ok=True)
        checkpoint_path = directory / filename
        metrics_path = checkpoint_path.with_suffix(".json")
        parent = None if name == "pets" else output / "theta_pets.pt"
        parent_hash = _file_hash(parent) if parent else None
        signature = {"cfg": run_cfg, "parent_sha256": parent_hash, "text_frozen": frozen}
        if checkpoint_path.exists() or metrics_path.exists():
            if checkpoint_path.exists() and metrics_path.exists():
                saved = json.loads(metrics_path.read_text())
                if saved["signature"] == signature and saved["checkpoint_sha256"] == _file_hash(checkpoint_path):
                    print(f"Reusing completed {name}", flush=True)
                    return saved["metrics"]
            raise FileExistsError(f"Incomplete or mismatched artifacts at {checkpoint_path}; use a new folder or move them aside.")
        print(f"\nTraining {name} from {parent or 'OpenAI pretrained weights'}", flush=True)
        seed_everything(0)
        model, _, _ = load_clip(str(parent) if parent else None, device=device, cfg=run_cfg)
        loader, ids, classes = trains[dataset]
        stats = train_model(model, loader, tokenizer, classes, cfg["prompts"], run_cfg, device, frozen)
        started = time.perf_counter()
        accuracy = evaluate_model(model, tests, tokenizer, cfg["prompts"], device)
        eval_seconds = time.perf_counter() - started
        metrics = {"run": name, "checkpoint": str(checkpoint_path),
                   "parent": str(parent) if parent else "openai", "dataset": dataset,
                   "lr": run_cfg["finetune"]["lr"], "epochs": run_cfg["finetune"]["epochs"],
                   "batch_size": run_cfg["finetune"]["batch_size"], "text_frozen": frozen,
                   "gpu": environment["gpu"], "device": device, **stats,
                   "eval_seconds": eval_seconds, **accuracy}
        pets_reference = rows[1]["OxfordIIITPet_accuracy_pct"] if name != "pets" else accuracy["OxfordIIITPet_accuracy_pct"]
        metrics["pets_forgetting_pp"] = pets_reference - accuracy["OxfordIIITPet_accuracy_pct"]
        model.cpu()
        temporary = checkpoint_path.with_suffix(".pt.tmp")
        torch.save(model.state_dict(), temporary)
        temporary.replace(checkpoint_path)
        _json_write(metrics_path, {"signature": signature, "checkpoint_sha256": _file_hash(checkpoint_path),
                                   "metrics": metrics, "train_ids": ids})
        del model
        gc.collect()
        if device.startswith("cuda"):
            torch.cuda.empty_cache()
        return metrics

    def save_table() -> None:
        columns = list(dict.fromkeys(key for row in rows for key in row))
        temporary = output / "training_results.csv.tmp"
        with temporary.open("w", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=columns)
            writer.writeheader()
            writer.writerows(rows)
        temporary.replace(output / "training_results.csv")

    save_table()
    for spec in RUNS:
        rows.append(run_one(spec, cfg, output))
        save_table()
    drop = rows[3]["pets_forgetting_pp"]
    status = {"base_pets_forgetting_pp": drop, "fallback_triggered": drop < 1.0,
              "base_ratio_defined": drop > 1.0,
              "note": "Sensitivity runs respond to a test metric; report separately as exploratory."}
    if drop < 1.0:
        print(f"Pets forgetting is {drop:.2f} pp (<1). A matched EuroSAT sensitivity pair is needed.", flush=True)
        if fallback_lr is not None:
            stronger = copy.deepcopy(cfg)
            stronger["finetune"]["lr"] = fallback_lr
            for spec in RUNS[2:]:
                rows.append(run_one(spec, stronger, output / "sensitivity_lr3e-5"))
                save_table()
            status["sensitivity_pets_forgetting_pp"] = rows[-2]["pets_forgetting_pp"]
            status["sensitivity_ratio_defined"] = rows[-2]["pets_forgetting_pp"] > 1.0
        else:
            print("Rerun with fallback_lr=3e-5 to add those runs; completed base runs will be reused.", flush=True)
    _json_write(output / "forgetting_check.json", status)
    return rows


In [ ]:
import torch
from src.common import load_config
from src.finetune import run_experiments

assert torch.cuda.is_available(), 'Select a GPU runtime, then run again.'
print('GPU:', torch.cuda.get_device_name(0))
print('Visible GPUs:', torch.cuda.device_count(), '| Training uses cuda:0 only')
cfg = load_config()
cfg['data']['root'] = str(DATA_DIR)
cfg['data']['num_workers'] = 2
print('Base training settings:', cfg['finetune'])
print('Prompt templates:', cfg['prompts'])


## Run the four experiments

This first evaluates pretrained CLIP, then trains and evaluates each checkpoint
on the same four test subsets. Each run prints progress and writes its checkpoint,
JSON metrics and `training_results.csv`. Float32 test evaluation matches Parts A/C.
Training uses mixed precision plus activation checkpointing to reduce GPU memory.

`train_minutes` is **GPU-synchronized elapsed training time**: it includes batch
loading, forward/backward, optimizer updates and the frozen control's prompt cache.
It excludes downloads, model loading, test evaluation, checksum verification and
checkpoint writes. It is a feasibility measurement on the named GPU, not
pure kernel time. `eval_seconds` and peak allocated GPU memory are also recorded.

Local session files are temporary. On Kaggle, save a completed notebook version
with outputs and verify the files appear in its Output tab before ending the
session. A draft/Quick Save of the code alone does not establish that artifacts
are preserved. On Colab, download artifacts or use the optional Drive destination.

Rerunning this cell reuses completed runs only when the protocol, installed package
versions, parent hash and checkpoint checksum match. An interrupted training run
restarts from its parent. Partial checkpoint files must be moved aside if reported.
For a new protocol, choose a new output directory. Keep the same runtime/package
versions when resuming. Do not rerun source-writing cells after importing modules;
restart the runtime after changing source.


In [ ]:
rows = run_experiments(cfg, str(OUTPUT_DIR), device='cuda')


## Measured table and low-forgetting check

Accuracy columns are percentages; `pets_forgetting_pp` is Pets accuracy at
`theta_pets.pt` minus Pets accuracy after the branch, in **percentage points**.
The pretrained row is evaluation only and has no training time.

If the full EuroSAT branch drops by **less than 1 point**, the next cell runs a
matched EuroSAT pair (both encoders / frozen text) at `3e-5`, each initialized from
the original Pets checkpoint. It preserves the four base checkpoints and writes
the additional two into `sensitivity_lr3e-5/`, logging the changed learning rate.
Otherwise it reuses the four existing runs. This is a conditional extra pair,
not part of the four-run budget. Report it as exploratory because a test metric
triggered it. Greater forgetting is not guaranteed; report small/negative drops
honestly. The Part C dominance ratio is undefined for a drop ≤1 point.


In [ ]:
rows = run_experiments(cfg, str(OUTPUT_DIR), device='cuda', fallback_lr=3e-5)

import json
import pandas as pd
table = pd.read_csv(OUTPUT_DIR / 'training_results.csv')
columns = ['run', 'lr', 'epochs', 'train_images', 'train_minutes',
           'OxfordIIITPet_accuracy_pct', 'Flowers102_accuracy_pct',
           'EuroSAT_accuracy_pct', 'DTD_accuracy_pct', 'pets_forgetting_pp']
display(table.reindex(columns=columns).round(3))
print(json.dumps(json.loads((OUTPUT_DIR / 'forgetting_check.json').read_text()), indent=2))
print('Base checkpoints:')
for file in sorted(OUTPUT_DIR.glob('theta_*.pt')):
    print(file.name, f'{file.stat().st_size / 1e6:.1f} MB')


## Download the artifacts

The source notebook belongs in GitHub. Large model checkpoints remain ignored by
Git; share the exported archive separately with the team. Kaggle automatically
creates `/kaggle/working/part_b_artifacts.zip` below, containing checkpoints,
tables and reproducibility metadata. After a successful **Save & Run All**, open
the saved version's **Output** tab and download that archive. Confirm it exists
before stopping the session. Dataset archives are excluded from this export.

On Colab, set `DOWNLOAD_ARTIFACTS=True` below to create and download the archive.
It can be several GB; allow the download to finish. If Drive is enabled, files
also persist at `OUTPUT_DIR`. The export checks that all four base checkpoints,
metrics and freeze checks exist; it is not a substitute for Part C analysis.


In [ ]:
# Validate completion before exporting. This does not train or load models.
import json
from src.finetune import RUNS
for _, _, filename, frozen in RUNS:
    checkpoint = OUTPUT_DIR / filename
    assert checkpoint.is_file(), f'Missing checkpoint: {checkpoint}'
    metadata = json.loads(checkpoint.with_suffix('.json').read_text())
    metrics = metadata['metrics']
    assert metrics['logit_scale_frozen_verified'] is True
    if frozen:
        assert metrics['text_frozen_verified'] is True
status = json.loads((OUTPUT_DIR / 'forgetting_check.json').read_text())
if status['fallback_triggered']:
    sensitivity = OUTPUT_DIR / 'sensitivity_lr3e-5'
    for filename in ('theta_pets_eurosat.pt', 'theta_pets_eurosat_textfrozen.pt'):
        assert (sensitivity / filename).is_file(), 'Finish the sensitivity cell before exporting.'
for filename in ('training_results.csv', 'protocol.json', 'environment.json'):
    assert (OUTPUT_DIR / filename).is_file(), f'Missing {filename}'
print('All required base files and freeze checks are present.')
print('Recorded source fingerprint:', json.loads((OUTPUT_DIR / 'protocol.json').read_text())['code_sha256'])

DOWNLOAD_ARTIFACTS = IS_KAGGLE  # Kaggle exports automatically; enable manually on Colab.
if DOWNLOAD_ARTIFACTS:
    import zipfile
    archive = EXPORT_DIR / 'part_b_artifacts.zip'
    with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_STORED) as bundle:
        for artifact in sorted(OUTPUT_DIR.rglob('*')):
            if artifact.is_file() and not artifact.name.endswith('.tmp'):
                bundle.write(artifact, Path('checkpoints') / artifact.relative_to(OUTPUT_DIR))
    if IS_COLAB:
        from google.colab import files
        files.download(str(archive))
    else:
        print('Artifact archive:', archive)
        print('On Kaggle, save the completed version and download this file from Output.')
else:
    print('Artifacts are at', OUTPUT_DIR)
    print('Enable DOWNLOAD_ARTIFACTS to export them before this session expires.')


## Demo, slides, report and viva

**3–4 minute screen recording (Thursday; Shikshith narrates):**

- 0:00–0:35: show GPU, CLIP model, four-run scope, prompts and split rule.
- 0:35–1:20: show the objective, AdamW settings and frozen `logit_scale`.
- 1:20–2:05: show a live run's progress, checkpoint saving and measured time.
- 2:05–3:05: show the completed table, compare Pets→Flowers and Pets→EuroSAT,
  then explain the frozen-text control and low-forgetting check.
- 3:05–3:40: show saved artifacts and the Phase-II timeline. Training may take
  longer than the video; prepare completed results before recording and show
  live progress separately. Do not present reused runs as newly measured runs.

**Implementation slide:** the two branches from Pets, loss/settings, notebook
progress screenshot, results-table screenshot, artifact location and video link.

**Timeline / Phase-II slide:** Phase I implements two pairs plus a text-frozen
control; Phase II expands to the promised 56-pair grid, repeated seeds and
uncertainty analysis. The four current datasets provide 12 ordered pairs; a
56-pair grid requires eight datasets. Confirm the remaining four in Phase II.
Leave dates tied to the team's agreed schedule rather than inventing them.

**Feasibility box:** public torchvision datasets, public OpenAI CLIP weights and
public project code; GPU model and **measured** minutes per run from the table.
The signal uses forward passes only, with no optimization. Put “minutes” on the
slide only after Part A measures its runtime; this notebook does not measure it.

**Report — Implementation:** describe CLIP, train/test manifests, prompt ensemble,
cross-entropy, AdamW, precision, frozen scale, branches and the control. Describe
EuroSAT's custom holdout explicitly. Include versions from `environment.json`.

**Report — Results:** paste the measured table. State Pets accuracy before and
after each shift and the difference in points; report target-domain accuracy and
GPU time. Separate sensitivity runs from base runs. The close/far designations
are experimental hypotheses, not results. Do not infer encoder contributions
from accuracy alone; Part C supplies the swap/Shapley measurements.

**Viva:** “We optimize image-to-all-class cross-entropy through the image and text
encoders using ten-prompt class prototypes. AdamW uses lr 1e-5, batch 64 and one
epoch; the learned temperature stays frozen. Each branch reloads the Pets
checkpoint. The frozen-text run holds language parameters fixed, so a change in
Pets predictions must come from vision; Part C should give zero text contribution
up to numerical tolerance. Training took **[measured minutes]** on **[GPU]**.”

Sources: [Pets splits](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.OxfordIIITPet.html),
[EuroSAT API](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.EuroSAT.html),
[Flowers label order](https://github.com/pytorch/vision/blob/main/torchvision/datasets/flowers102.py),
[OpenCLIP](https://github.com/mlfoundations/open_clip).
